## **ЗПАД | Лабораторна робота 2**
**Виконав: Пономаренко Роман Володимирович | ФБ-44**

### **Частина 2**

Здійснити data cleaning

In [1]:
import pandas as pd

df_power = pd.read_csv('household_power_consumption.txt', sep=';', na_values=['?'], low_memory=False)
df_power.dropna(inplace=True)

df_power['Date'] = pd.to_datetime(df_power['Date'], format='%d/%m/%Y')
df_power['Time'] = pd.to_datetime(df_power['Time'], format='%H:%M:%S').dt.time

numeric_cols = [
    'Global_active_power', 'Global_reactive_power', 'Voltage', 
    'Global_intensity', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3'
]
df_power[numeric_cols] = df_power[numeric_cols].astype(float)

print("Датасет готовий. Розмір:", df_power.shape)
print(df_power.head(), "\n")

Датасет готовий. Розмір: (2049280, 9)
        Date      Time  Global_active_power  Global_reactive_power  Voltage  \
0 2006-12-16  17:24:00                4.216                  0.418   234.84   
1 2006-12-16  17:25:00                5.360                  0.436   233.63   
2 2006-12-16  17:26:00                5.374                  0.498   233.29   
3 2006-12-16  17:27:00                5.388                  0.502   233.74   
4 2006-12-16  17:28:00                3.666                  0.528   235.68   

   Global_intensity  Sub_metering_1  Sub_metering_2  Sub_metering_3  
0              18.4             0.0             1.0            17.0  
1              23.0             0.0             1.0            16.0  
2              23.0             0.0             2.0            17.0  
3              23.0             0.0             1.0            17.0  
4              15.8             0.0             1.0            17.0   



Окремими функціями сформувати вибірки:
* Обрати всі записи, у яких загальна активна споживана потужність перевищує 5 кВт.
* Обрати всі записи, у яких сила струму лежить в межах 19-20 А, для них виявити ті, у яких пральна машина та холодильних споживають більше, ніж бойлер та кондиціонер.
* Обрати випадковим чином 500000 записів (без повторів елементів вибірки), для них обчислити середні величини усіх 3-х груп споживання електричної енергії
* Обрати ті записи, які після 18-00 споживають понад 6 кВт за хвилину в середньому, серед відібраних визначити ті, у яких основне споживання електроенергії у вказаний проміжок часу припадає на пральну машину, сушарку, холодильник та освітлення (група 2 є найбільшою), а потім обрати кожен третій результат із першої половини та кожен четвертий результат із другої половини.


In [2]:
import pandas as pd
import datetime
import timeit

# Потужність > 5 кВт
def get_high_power():
    return df_power[df_power['Global_active_power'] > 5.0]

res_1 = None
def wrap_1():
    global res_1
    res_1 = get_high_power()

time_1 = timeit.timeit(wrap_1, number=1)
print(f"Вибірка 1 (> 5 кВт)")
print(f"Час виконання: {time_1:.4f} сек | Знайдено: {len(res_1)}")
print(res_1.head(), "\n")

# Сила струму 19-20 А, пральна та холодильник (2) > бойлер та конд (3)
def get_laundry_over_boiler():
    mask_current = df_power['Global_intensity'].between(19, 20)
    mask_power = df_power['Sub_metering_2'] > df_power['Sub_metering_3']
    return df_power[mask_current & mask_power]

res_2 = None
def wrap_2():
    global res_2
    res_2 = get_laundry_over_boiler()

time_2 = timeit.timeit(wrap_2, number=1)
print(f"Вибірка 2 (19-20 А, пральна+холодильник > бойлер)")
print(f"Час виконання: {time_2:.4f} сек | Знайдено: {len(res_2)}")
print(res_2.head(), "\n")

# Випадкові 500k записів і середнє
def get_random_sample_means():
    sample = df_power.sample(n=500000, replace=False)
    return sample[['Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']].mean()

res_3 = None
def wrap_3():
    global res_3
    res_3 = get_random_sample_means()

time_3 = timeit.timeit(wrap_3, number=1)
print(f"Вибірка 3 (Середнє для випадкових 500k записів)")
print(f"Час виконання: {time_3:.4f} сек")
print(res_3, "\n")

# Після 18:00, > 6 кВт за хв, Sub2 найбільша. Зрізи.
def get_evening_complex():
    time_mask = df_power['Time'] > datetime.time(18, 0, 0)
    power_mask = df_power['Global_active_power'] > 6
    sub2_max_mask = (df_power['Sub_metering_2'] > df_power['Sub_metering_1']) & \
                    (df_power['Sub_metering_2'] > df_power['Sub_metering_3'])
    
    filtered = df_power[time_mask & power_mask & sub2_max_mask]
    
    if filtered.empty:
        return filtered
        
    half_idx = len(filtered) // 2
    first_half = filtered.iloc[:half_idx:3]
    second_half = filtered.iloc[half_idx::4]
    
    return pd.concat([first_half, second_half])

res_4 = None
def wrap_4():
    global res_4
    res_4 = get_evening_complex()

time_4 = timeit.timeit(wrap_4, number=1)
print(f"Вибірка 4 (Складна після 18:00)")
print(f"Час виконання: {time_4:.4f} сек | Знайдено після зрізу: {len(res_4)}")
print(res_4.head(), "\n")

Вибірка 1 (> 5 кВт)
Час виконання: 0.0042 сек | Знайдено: 17547
         Date      Time  Global_active_power  Global_reactive_power  Voltage  \
1  2006-12-16  17:25:00                5.360                  0.436   233.63   
2  2006-12-16  17:26:00                5.374                  0.498   233.29   
3  2006-12-16  17:27:00                5.388                  0.502   233.74   
11 2006-12-16  17:35:00                5.412                  0.470   232.78   
12 2006-12-16  17:36:00                5.224                  0.478   232.99   

    Global_intensity  Sub_metering_1  Sub_metering_2  Sub_metering_3  
1               23.0             0.0             1.0            16.0  
2               23.0             0.0             2.0            17.0  
3               23.0             0.0             1.0            17.0  
11              23.2             0.0             1.0            17.0  
12              22.4             0.0             1.0            16.0   

Вибірка 2 (19-20 А, пральна

Пронормувати та стандартизувати вибраний датасет

In [3]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler

cols_to_scale = ['Global_active_power', 'Global_intensity']
df_scaled = df_power[cols_to_scale].copy()

# Нормалізація (MinMax)
scaler_minmax = MinMaxScaler()
df_norm = pd.DataFrame(
    scaler_minmax.fit_transform(df_scaled), 
    columns=[f"{c}_norm" for c in cols_to_scale],
    index=df_power.index # зберігаємо оригінальну нумерацію рядків
)

# Стандартизація (Standard)
scaler_std = StandardScaler()
df_std = pd.DataFrame(
    scaler_std.fit_transform(df_scaled), 
    columns=[f"{c}_std" for c in cols_to_scale],
    index=df_power.index # і тут теж
)

print("Нормалізовані дані (від 0 до 1):")
print(df_norm.head(), "\n")

print("Стандартизовані дані (центровані біля 0):")
print(df_std.head())

Нормалізовані дані (від 0 до 1):
   Global_active_power_norm  Global_intensity_norm
0                  0.374796               0.377593
1                  0.478363               0.473029
2                  0.479631               0.473029
3                  0.480898               0.473029
4                  0.325005               0.323651 

Стандартизовані дані (центровані біля 0):
   Global_active_power_std  Global_intensity_std
0                 2.955077              3.098789
1                 4.037085              4.133800
2                 4.050326              4.133800
3                 4.063567              4.133800
4                 2.434881              2.513782


Підрахувати коефіцієнт Пірсона та Спірмена для двох integer/real атрибутів.

In [4]:
pearson_corr = df_power['Global_active_power'].corr(df_power['Global_intensity'], method='pearson')
spearman_corr = df_power['Global_active_power'].corr(df_power['Global_intensity'], method='spearman')

print(f"Коефіцієнт Пірсона: {pearson_corr:.4f}")
print(f"Коефіцієнт Спірмена: {spearman_corr:.4f}")

Коефіцієнт Пірсона: 0.9989
Коефіцієнт Спірмена: 0.9954


Провести One Hot Encoding категоріального атрибута.

In [5]:
# Створюємо категорію "День тижня"
df_power['DayOfWeek'] = df_power['Date'].dt.day_name()

# Робимо OHE
df_encoded = pd.get_dummies(df_power, columns=['DayOfWeek'])

# Виводимо тільки нові колонки, щоб показати, що вийшло
ohe_cols = [col for col in df_encoded.columns if 'DayOfWeek' in col]
print("Результат One Hot Encoding для днів тижня:")
print(df_encoded[ohe_cols].head())

Результат One Hot Encoding для днів тижня:
   DayOfWeek_Friday  DayOfWeek_Monday  DayOfWeek_Saturday  DayOfWeek_Sunday  \
0             False             False                True             False   
1             False             False                True             False   
2             False             False                True             False   
3             False             False                True             False   
4             False             False                True             False   

   DayOfWeek_Thursday  DayOfWeek_Tuesday  DayOfWeek_Wednesday  
0               False              False                False  
1               False              False                False  
2               False              False                False  
3               False              False                False  
4               False              False                False  
